# 01 — Ingest
Load the hand-collected Taskmaster points CSV from `data/raw/` into DuckDB, untouched.

This project's data was hand-collected (total points per contestant per series, S1–S21) rather than scraped, so there is no network fetch. The raw file is read as-is and loaded into the project database as `contestant_points_raw`, with provenance recorded in `_sources`.

Nothing here modifies data — cleaning happens in `02-clean`.

In [ ]:
import sys, os
from pathlib import Path

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

from src.ingest import load_config, load_raw_csv
from src.clean_quality import get_connection, load_to_duckdb, register_source, get_sources

cfg = load_config('config.yaml')
con = get_connection(cfg)
print(f'Project: {cfg["project_name"]}')

## Read the raw CSV
The file `data/raw/taskmaster_data.csv` was placed here by hand. Columns: `Series Number`, `Contestant Name`, `Total Points`.

In [ ]:
df_raw = load_raw_csv(cfg, 'taskmaster_data.csv')
print(df_raw.shape)
df_raw.head()

In [ ]:
# Sanity: expect 21 series, 5 contestants each = 105 rows
print('Rows          :', len(df_raw))
print('Series present:', sorted(df_raw['Series Number'].unique()))
print('Per-series counts:')
print(df_raw['Series Number'].value_counts().sort_index().to_string())

## Load into DuckDB (raw, untouched)

In [ ]:
load_to_duckdb(df_raw, 'contestant_points_raw', con)
print('Loaded contestant_points_raw:',
      con.execute('SELECT COUNT(*) FROM contestant_points_raw').fetchone()[0], 'rows')

## Register provenance
The critical caveat lives in `series_breaks`: episode counts differ across series, so raw totals are **not** comparable series-to-series — downstream we normalize to share of series points.

In [ ]:
register_source(
    con,
    table='contestant_points_raw',
    name='Taskmaster UK contestant total points by series (hand-collected)',
    url='local: data/raw/taskmaster_data.csv',
    license='Facts (scores) not copyrightable; compiled dataset by author.',
    notes='Total points per contestant per series, S1-S21 (5 per series, 105 rows). '
          'Excludes Champion of Champions / New Year Treat specials and Junior Taskmaster. '
          'Names corrected to official full names before ingest.',
    retrieved='2026-09-01',
    methodology='Cumulative points a contestant earned across all tasks in their series '
                '(prize + filmed + studio/live tasks), as adjudicated on-air by Greg Davies. '
                'Raw series total, not per-episode normalized.',
    series_breaks='Episode count varies by series: S1=6, S2-3=5, S4-5=8, S6-21=10. '
                  'More episodes = more tasks = more points available, so RAW totals are '
                  'NOT comparable across series. Compare on share of series points instead.',
)
get_sources(con)

---
**Next:** `02-clean.ipynb` — cast/trim/validate inside DuckDB, quality report, save interim Parquet.

---
## Cleanup
Close the DuckDB connection so the lock is released for other tools (DBCode, other notebooks). Runs on “Run All”.

In [ ]:
con.close()
print('connection closed')